# pix2pix Training — Real Face to Comic

This notebook trains a **pix2pix** conditional GAN to translate real face images into comic-style images.

**Architecture:**
- Generator: U-Net with skip connections
- Discriminator: 70x70 PatchGAN
- Loss: Adversarial (BCE) + L1 reconstruction

**Hardware:** Automatically uses CUDA (NVIDIA GPU), MPS (Apple Silicon), or CPU.


In [10]:
import os
import sys
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt

# Let notebooks import from src/
sys.path.insert(0, str(Path("..").resolve()))
from src.dataset import FaceComicDataset


## 1. Configuration

All hyperparameters in one place. Change these to tune training.


In [11]:
# ── Hyperparameters ──
IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 2
LR = 2e-4
BETAS = (0.5, 0.999)
LAMBDA_L1 = 100
# Dataset class lives in src/dataset.py so multiprocessing workers can import it.
NUM_WORKERS = min(4, os.cpu_count() or 1)

# ── Paths ──
DATA_DIR = Path("../data/Input")
REAL_DIR = DATA_DIR / "real"
COMIC_DIR = DATA_DIR / "comic"
CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_EVERY = 10

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ──
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device:      {DEVICE}")
print(f"Workers:     {NUM_WORKERS}")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Epochs:      {NUM_EPOCHS}")


Device:      mps
Workers:     4
Batch size:  16
Epochs:      2


## 2. Dataset

Each sample is a paired `(real, comic)` image. Both are resized to `256x256` and normalized to `[-1, 1]`.


In [12]:
dataset = FaceComicDataset(REAL_DIR, COMIC_DIR, image_size=IMAGE_SIZE)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

print(f"Dataset size:  {len(dataset):,} pairs")
print(f"Batches/epoch: {len(dataloader)}")


Dataset size:  10,000 pairs
Batches/epoch: 625


## 3. Generator — U-Net

Encoder-decoder with skip connections. The encoder downsamples to a bottleneck, the decoder upsamples back, and skip connections carry high-frequency detail across.


In [13]:
class UNetDown(nn.Module):
    def __init__(self, in_ch, out_ch, normalize=True):
        super().__init__()
        layers = [nn.Conv2d(in_ch, out_ch, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_ch))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetUp(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_ch, out_ch, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        ]
        if dropout:
            layers.append(nn.Dropout(0.5))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], dim=1)


class Generator(nn.Module):
    def __init__(self, in_ch=3, out_ch=3):
        super().__init__()
        self.d1 = UNetDown(in_ch, 64, normalize=False)
        self.d2 = UNetDown(64, 128)
        self.d3 = UNetDown(128, 256)
        self.d4 = UNetDown(256, 512)
        self.d5 = UNetDown(512, 512)
        self.d6 = UNetDown(512, 512)
        self.d7 = UNetDown(512, 512)
        self.d8 = UNetDown(512, 512, normalize=False)

        self.u1 = UNetUp(512, 512, dropout=True)
        self.u2 = UNetUp(1024, 512, dropout=True)
        self.u3 = UNetUp(1024, 512, dropout=True)
        self.u4 = UNetUp(1024, 512)
        self.u5 = UNetUp(1024, 256)
        self.u6 = UNetUp(512, 128)
        self.u7 = UNetUp(256, 64)

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_ch, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)
        d5 = self.d5(d4)
        d6 = self.d6(d5)
        d7 = self.d7(d6)
        d8 = self.d8(d7)

        u1 = self.u1(d8, d7)
        u2 = self.u2(u1, d6)
        u3 = self.u3(u2, d5)
        u4 = self.u4(u3, d4)
        u5 = self.u5(u4, d3)
        u6 = self.u6(u5, d2)
        u7 = self.u7(u6, d1)

        return self.final(u7)


## 4. Discriminator — PatchGAN

Classifies overlapping 70x70 patches as real or fake. This forces the generator to produce sharp, locally realistic textures everywhere in the image.


In [14]:
class Discriminator(nn.Module):
    def __init__(self, in_ch=6):
        super().__init__()
        def block(in_c, out_c, normalize=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
            if normalize:
                layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.model = nn.Sequential(
            *block(in_ch, 64, normalize=False),
            *block(64, 128),
            *block(128, 256),
            nn.Conv2d(256, 512, 4, 1, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, 4, 1, 1),
        )

    def forward(self, real_input, target_or_fake):
        x = torch.cat([real_input, target_or_fake], dim=1)
        return self.model(x)


## 5. Initialize Models and Optimizers


In [15]:
def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)


gen = Generator().to(DEVICE)
disc = Discriminator().to(DEVICE)
gen.apply(init_weights)
disc.apply(init_weights)

opt_gen = optim.Adam(gen.parameters(), lr=LR, betas=BETAS)
opt_disc = optim.Adam(disc.parameters(), lr=LR, betas=BETAS)

criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1 = nn.L1Loss()

gen_params = sum(p.numel() for p in gen.parameters())
disc_params = sum(p.numel() for p in disc.parameters())
print(f"Generator params:     {gen_params:,}")
print(f"Discriminator params: {disc_params:,}")


Generator params:     54,413,955
Discriminator params: 2,768,641


## 6. Training Loop

Standard pix2pix training:

1. **Discriminator step:** learn to tell real pairs from fake pairs.
2. **Generator step:** fool the discriminator + minimize L1 distance to the real comic target.

Checkpoints are saved every `SAVE_EVERY` epochs.


In [ ]:
history = {"d_loss": [], "g_loss": [], "epoch_time": []}

for epoch in range(1, NUM_EPOCHS + 1):
    gen.train()
    disc.train()
    d_epoch_loss = 0.0
    g_epoch_loss = 0.0
    start = time.time()

    for real_img, comic_img in dataloader:
        real_img = real_img.to(DEVICE)
        comic_img = comic_img.to(DEVICE)
        fake_comic = gen(real_img)

        # ── Train Discriminator ──
        disc_real = disc(real_img, comic_img)
        disc_fake = disc(real_img, fake_comic.detach())
        real_label = torch.ones_like(disc_real, device=DEVICE)
        fake_label = torch.zeros_like(disc_fake, device=DEVICE)

        loss_d_real = criterion_gan(disc_real, real_label)
        loss_d_fake = criterion_gan(disc_fake, fake_label)
        loss_d = (loss_d_real + loss_d_fake) * 0.5

        opt_disc.zero_grad()
        loss_d.backward()
        opt_disc.step()

        # ── Train Generator ──
        disc_fake_for_g = disc(real_img, fake_comic)
        loss_g_gan = criterion_gan(
            disc_fake_for_g,
            torch.ones_like(disc_fake_for_g, device=DEVICE),
        )
        loss_g_l1 = criterion_l1(fake_comic, comic_img) * LAMBDA_L1
        loss_g = loss_g_gan + loss_g_l1

        opt_gen.zero_grad()
        loss_g.backward()
        opt_gen.step()

        d_epoch_loss += loss_d.item()
        g_epoch_loss += loss_g.item()

    n_batches = len(dataloader)
    d_avg = d_epoch_loss / n_batches
    g_avg = g_epoch_loss / n_batches
    elapsed = time.time() - start

    history["d_loss"].append(d_avg)
    history["g_loss"].append(g_avg)
    history["epoch_time"].append(elapsed)

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS}"
        f" | D loss: {d_avg:.4f}"
        f" | G loss: {g_avg:.4f}"
        f" | Time: {elapsed:.1f}s"
    )

    if epoch % SAVE_EVERY == 0 or epoch == NUM_EPOCHS:
        ckpt = {
            "epoch": epoch,
            "gen_state": gen.state_dict(),
            "disc_state": disc.state_dict(),
            "opt_gen_state": opt_gen.state_dict(),
            "opt_disc_state": opt_disc.state_dict(),
            "history": history,
        }
        path = CHECKPOINT_DIR / f"pix2pix_epoch_{epoch:03d}.pt"
        torch.save(ckpt, path)
        print(f"  Saved checkpoint: {path}")

print("Training complete.")


## 7. Loss Curves


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["d_loss"], label="Discriminator")
axes[0].plot(history["g_loss"], label="Generator")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()

axes[1].plot(history["epoch_time"])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Seconds")
axes[1].set_title("Time per Epoch")

plt.tight_layout()
plt.show()


## 8. Visual Results

Show a few real images alongside the generator output and the ground-truth comic target.


In [ ]:
gen.eval()
sample_real, sample_comic = next(iter(dataloader))
sample_real = sample_real.to(DEVICE)

with torch.no_grad():
    sample_fake = gen(sample_real)


def to_numpy(tensor):
    return (tensor.cpu().numpy().transpose(0, 2, 3, 1) * 0.5 + 0.5).clip(0, 1)


real_np = to_numpy(sample_real)
comic_np = to_numpy(sample_comic)
fake_np = to_numpy(sample_fake)

n_show = min(4, len(real_np))
fig, axes = plt.subplots(3, n_show, figsize=(3.5 * n_show, 10))
row_titles = ["Real Input", "Generated Comic", "Ground Truth"]

for i in range(n_show):
    axes[0, i].imshow(real_np[i])
    axes[1, i].imshow(fake_np[i])
    axes[2, i].imshow(comic_np[i])
    for row in range(3):
        axes[row, i].axis("off")

for row, title in enumerate(row_titles):
    axes[row, 0].set_title(title, fontsize=12)

plt.tight_layout()
plt.show()
